# Chess Neural Network — Training Pipeline

Auto-detects training phase based on available checkpoints.
Runs on Kaggle T4/P100 GPU.

In [ ]:
# Clone the repo and install dependencies
!git clone https://github.com/amiohossain/Chess.git /kaggle/working/Chess
!pip install -q -r /kaggle/working/Chess/requirements.txt

In [ ]:
import os
import sys
import glob
import subprocess

sys.path.insert(0, '/kaggle/working/Chess')

DATA_DIR = '/kaggle/input/chess-training-data'
WORKING_DIR = '/kaggle/working/Chess'
HDF5_PATH = '/kaggle/working/supervised_positions.h5'
os.makedirs('/kaggle/working/data', exist_ok=True)

# Check if HDF5 already exists in the input dataset
h5_files = glob.glob(f'{DATA_DIR}/*.h5')
if h5_files:
    print(f'Found existing HDF5 data: {h5_files}')
    !cp {h5_files[0]} {HDF5_PATH}
else:
    print('No HDF5 found. Processing PGN.zst files...')
    zst_files = sorted(glob.glob(f'{DATA_DIR}/*.pgn.zst'))
    print(f'Found {len(zst_files)} PGN.zst files')

    # Decompress all PGNs, concatenate, then process once
    combined_pgn = '/kaggle/working/combined.pgn'

    for i, zst in enumerate(zst_files):
        pgn_part = f'/kaggle/working/temp_{i}.pgn'
        print(f'[{i+1}/{len(zst_files)}] Decompressing {os.path.basename(zst)}...')
        subprocess.run(['zstd', '-d', '-f', zst, '-o', pgn_part], check=True)

        # Concatenate to combined file
        with open(pgn_part, 'r') as src:
            with open(combined_pgn, 'a' if i > 0 else 'w') as dst:
                dst.write(src.read())
        os.remove(pgn_part)

    print(f'Combined PGN size: {os.path.getsize(combined_pgn) / 1e6:.0f} MB')

    # Process combined PGN to HDF5
    from src.data.pgn_processor import process_pgn
    n = process_pgn(
        pgn_path=combined_pgn,
        output_path=HDF5_PATH,
        max_positions=5_000_000,
        min_elo=2200,
    )
    os.remove(combined_pgn)
    print(f'\nHDF5 created with {n} positions')

In [ ]:
# Run the training pipeline (auto-detects phase)
from src.kaggle.kaggle_main import run_session
run_session()